<a href="https://colab.research.google.com/github/kea-S/llmQuantization/blob/main/cs4262_5462_efficient_ai_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
%pip install -q transformers==5.0.0 "trl[peft]==0.28.0" bitsandbytes==0.49.2 evaluate==0.4.6 google-genai tenacity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00


# 1. LLM Quantization

In the first part of this tutorial, we will explore the impact of quantization on performance and output quality.

We will use the [bitsandbytes](https://huggingface.co/docs/transformers/quantization/bitsandbytes) library, which provides quantization tools for LLMs.

First, import the dependencies.

In [2]:
import gc
import pprint
import random
import re
from typing import Any

import evaluate
import numpy as np
import torch
import tqdm
from datasets import Features, Value, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig,
    pipeline,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    torch_dtype = torch.float32
print(f"Using torch dtype: {torch_dtype}")

Using device: cuda
Using torch dtype: torch.bfloat16


## 1.1 Prepare the dataset

### 1.1.1 Load the dataset

Throughout this tutorial, we will use a subset of OpenAI's [GSM8K](https://huggingface.co/datasets/openai/gsm8k) dataset, which contains diverse grade school math word problems that require multi-step reasoning.

Let's download and inspect the dataset

In [3]:
dataset = load_dataset("openai/gsm8k", name="main").shuffle(seed=SEED)
train_dataset = dataset["train"].select(range(300))
test_dataset = dataset["test"].select(range(30))

print("Columns:", train_dataset.column_names)
sample = train_dataset[0]
print("-" * 50)
print("Example question:")
print(sample["question"])
print("-" * 50)
print("Example answer:")
print(sample["answer"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Columns: ['question', 'answer']
--------------------------------------------------
Example question:
Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?
--------------------------------------------------
Example answer:
Mimi has 2 x 12 = <<2*12=24>>24 sea shells.
Kyle has 24 x 2 = <<24*2=48>>48 sea shells.
Leigh has 48 / 3 = <<48/3=16>>16 sea shells.
#### 16


### 1.1.2 Transform the dataset (10pt)

The GSM8K dataset originally contains the `question` and `answer` columns, which are not directly usable on an instruction-tuned LLM.

In this part, we will transform the `question` and `answer` columns into the `prompt` and `ground_truth` columns, which can be used for SFT later in this tutorial. The `prompt` and `ground_truth` columns should have the following format:
- Each row in the `prompt` column contains a list of messages of the following form: `[{"role": "system", "content": "..."}, {"role": "user", "content": "..."}]`
- Each row in the `ground_truth` column contains the final answer to the question as a string (e.g., "16").

**NOTE:**
- Each item in the `prompt` and `ground_truth` column is a list of dicts and a string, respectively.
- The final answer is contained in each row in the `answer` column, prefixed by "#### ".

**TODO:**
1. Implement the `to_prompt_ground_truth_pair` function to transform the question and answer into LLM prompts and ground-truth answer. The content of each column is described above.

In [4]:
system_prompt = "You are a helpful assistant for solving math word problems."
user_prompt_template = """Answer the following question by explaining your reasoning very concisely and then end your answer with the final numerical answer prefixed by "#### ".

{question}
"""


def extract_final_answer(answer: str) -> str:
    """Extract the final answer from the model's output

    The output is expected to contain a final answer prefixed by "#### ".
    """
    match = re.search(r"#### (\S+)", answer)
    if match:
        return match.group(1).strip()
    return answer.strip()


def to_prompt_ground_truth_pair(sample: dict[str, Any]) -> dict[str, Any]:
    """Convert a sample from the dataset into a prompt-ground truth pair by transforming
    the `question` and `answer` fields

    See the above description for the expected format of the prompt and ground truth.
    """
    prompt = []
    ground_truth = []
    # TODO: Add your code here
    # ==== Start of your code ====
    prompt.append({"role": "system", "content": system_prompt})
    prompt.append(
        {"role": "user", "content": user_prompt_template.format(question=sample["question"])},
    )
    ground_truth = extract_final_answer(sample["answer"])
    # ==== End of your code ====
    return {"prompt": prompt, "ground_truth": ground_truth}


features = Features(
    {
        "question": Value("string"),
        "answer": Value("string"),
        "prompt": [{"role": Value("string"), "content": Value("string")}],
        "ground_truth": Value("string"),
    }
)
train_dataset = train_dataset.map(to_prompt_ground_truth_pair, features=features)
test_dataset = test_dataset.map(to_prompt_ground_truth_pair, features=features)

sample = train_dataset[0]
print("Example prompt:")
pprint.pprint(sample["prompt"])
print("-" * 50)
print("Example ground truth:")
pprint.pprint(sample["ground_truth"])


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Example prompt:
[{'content': 'You are a helpful assistant for solving math word problems.',
  'role': 'system'},
 {'content': 'Answer the following question by explaining your reasoning very '
             'concisely and then end your answer with the final numerical '
             'answer prefixed by "#### ".\n'
             '\n'
             'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice '
             'as many shells as Mimi and put them in his pocket. Leigh grabbed '
             'one-third of the shells that Kyle found.  How many seashells did '
             'Leigh have?\n',
  'role': 'user'}]
--------------------------------------------------
Example ground truth:
'16'


## 1.2 Quantize model

We will use the [Qwen/Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) model throughout this tutorial.

In [5]:
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_name, padding_side="left")
# Set the default generation parameters
GEN_BATCH_SIZE = 16  # NOTE: You can adjust this based on your GPU memory
GEN_CONFIG = GenerationConfig(
    max_new_tokens=512,
    do_sample=False,
    temperature=0,  # You can ignore the invalid flag warning
    num_return_sequences=1,
    use_cache=True,
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


### 1.2.1 Inspect the original model

Check the memory footprint and output quality of the model before quantization

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch_dtype, device_map="auto"
)
print(f"Memory footprint: {model.get_memory_footprint() / (1024**3):.2f} GB")

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Memory footprint: 2.88 GB


In [8]:
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
sample = test_dataset[0]
prompt = sample["prompt"]
output = generator(prompt, generation_config=GEN_CONFIG)
output_content = output[0]["generated_text"][-1]["content"]
print(f"Prompt:\n{prompt}")
print("-" * 50)
print(f"Output:\n{output_content}")
print("-" * 50)
print(f"Extracted answer: {extract_final_answer(output_content)}")
print("-" * 50)
print(f"Ground truth:\n{sample['ground_truth']}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prompt:
[{'role': 'system', 'content': 'You are a helpful assistant for solving math word problems.'}, {'role': 'user', 'content': 'Answer the following question by explaining your reasoning very concisely and then end your answer with the final numerical answer prefixed by "#### ".\n\nDarrell and Allen\'s ages are in the ratio of 7:11. If their total age now is 162, calculate Allen\'s age 10 years from now.\n'}]
--------------------------------------------------
Output:
Let's denote Darrell's age as \(7x\) and Allen's age as \(11x\). According to the problem, their combined age is 162, so we have:
\[7x + 11x = 162\]
Simplifying this equation gives us:
\[18x = 162\]
Dividing both sides by 18 yields:
\[x = 9\]
Therefore, Darrell's current age is \(7 \times 9 = 63\) and Allen's current age is \(11 \times 9 = 99\).

To find out how old Allen will be 10 years from now, we add 10 to his current age:
\[99 + 10 = 109\]

So, Allen's age 10 years from now will be **109**.
----------------------

In [9]:
# Free up memory
del generator
del model
gc.collect()
torch.cuda.empty_cache()

### 1.2.2 Evaluate the quantized model

Check the memory footprint and output quality of the model after quantization.

We will use 4-bit bitsandbytes quantization for simplicity. For other quantization techniques, please refer to [Quantization](https://huggingface.co/docs/peft/en/developer_guides/quantization).

In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    dtype=torch_dtype,
    device_map="auto",
)
print(f"Memory footprint: {model.get_memory_footprint() / (1024**3):.2f} GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Memory footprint: 1.05 GB


In [11]:
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
sample = test_dataset[0]
prompt = sample["prompt"]
output = generator(prompt, generation_config=GEN_CONFIG)
output_content = output[0]["generated_text"][-1]["content"]
print(f"Prompt:\n{prompt}")
print("-" * 50)
print(f"Output:\n{output_content}")
print("-" * 50)
print(f"Extracted answer: {extract_final_answer(output_content)}")
print("-" * 50)
print(f"Ground truth:\n{sample['ground_truth']}")

Prompt:
[{'role': 'system', 'content': 'You are a helpful assistant for solving math word problems.'}, {'role': 'user', 'content': 'Answer the following question by explaining your reasoning very concisely and then end your answer with the final numerical answer prefixed by "#### ".\n\nDarrell and Allen\'s ages are in the ratio of 7:11. If their total age now is 162, calculate Allen\'s age 10 years from now.\n'}]
--------------------------------------------------
Output:
Let's denote Darrell's age as \(7x\) and Allen's age as \(11x\). According to the problem, their combined age is 162, so we have:
\[7x + 11x = 162\]
Simplifying this equation gives us:
\[18x = 162\]
Dividing both sides by 18 yields:
\[x = 9\]
Therefore, Darrell's current age is \(7 \times 9 = 63\) and Allen's current age is \(11 \times 9 = 99\).

To find out how old Allen will be 10 years from now, we add 10 to his current age:
\[99 + 10 = 109\]

So, Allen's age 10 years from now will be **109**.
----------------------

Evaluate the quantized model on the test dataset.

In [12]:
outputs: list[str] = []
for i in tqdm.tqdm(range(0, len(test_dataset), GEN_BATCH_SIZE)):
    batch = test_dataset[i : i + GEN_BATCH_SIZE]
    prompts = batch["prompt"]
    output = generator(prompts, generation_config=GEN_CONFIG)
    outputs.extend([o[0]["generated_text"][-1]["content"] for o in output])

100%|██████████| 2/2 [05:45<00:00, 172.87s/it]


In [13]:
answers = [extract_final_answer(output) for output in outputs]
em = evaluate.load("exact_match")
em_score = em.compute(predictions=answers, references=test_dataset["ground_truth"])
print(f"Exact Match: {em_score['exact_match']:.2f}")

Exact Match: 0.23


In [14]:
# Free up memory
del generator
del model
gc.collect()
torch.cuda.empty_cache()

# 2. Knowledge Distillation

In this part, we will distill a large teacher model ([gemma-3-27b-it](https://ai.google.dev/gemma/docs/core)) into the quantized student model.

Although Gemma 3 itself is open-source, the 27B model is too large to run on Google Colab, so we will access the model through Google's Gemini API. To enable this, you have to create an API key on [Google AI Studio](https://aistudio.google.com/api-keys) (or use an existing key) and configure Colab's _Secrets_. **Do not put your API key in the notebook itself.**

Since we do not have access to teacher logits (assuming), we will perform _black-box_ distillation.

In [15]:
import os
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed,
)  # You may use these to parallelize API calls

from datasets import Dataset, DatasetDict
from google import genai
from google.colab import userdata
from google.genai.types import Content, GenerateContentConfig, Part
from peft import AutoPeftModelForCausalLM, LoraConfig, prepare_model_for_kbit_training
from tenacity import retry, retry_if_exception_type, wait_exponential
from trl import SFTConfig, SFTTrainer

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

## 2.1 Build distilled dataset (20pt)

We will query the teacher on a subset of GSM8K and convert each sample into chat-formatted `prompt` and `completion` columns for `SFTTrainer`.

**TODOs:**
1. Implement the `create_distilled_dataset` function by querying the teacher model with each question in the given dataset. If you find the execution too slow, you may use `ThreadPoolExecutor` to parallelize API calls.
2. (Optional) Filter the distilled training data by the quality of the teacher's output.

**HINT:**
- You may use the provided helper functions for querying the Gemma model through Gemini API.
- You may apply additional filtering of the training data based on the correctness of the teacher's final answer.

In [17]:
@retry(
    reraise=True,
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type(Exception),
)
def query_teacher_once(
    client: genai.Client, question: str, generation_config: GenerationConfig
) -> str:
    contents = [
        Content(
            role="user",
            parts=[Part(text=user_prompt_template.format(question=question))],
        )
    ]
    response = client.models.generate_content(
        model="gemma-3-27b-it",
        contents=contents,
        config=GenerateContentConfig(
            candidate_count=generation_config.num_return_sequences,
            max_output_tokens=generation_config.max_new_tokens,
            temperature=(
                0
                if generation_config.do_sample is False
                else generation_config.temperature
            ),
            top_p=generation_config.top_p,
            top_k=generation_config.top_k,
        ),
    )
    try:
        return response.candidates[0].content.parts[0].text
    except Exception as e:
        print(f"WARN: error parsing teacher model response: {repr(e)}")
        print(f"WARN: full response: {response}")
        raise


def query_teacher(
    client: genai.Client, question: str, generation_config: GenerationConfig
) -> str:
    """Query the teacher model with retries and error handling"""
    try:
        return query_teacher_once(client, question, generation_config)
    except Exception:
        print(f"ERROR: failed to query teacher for question: {repr(question)}")
        raise


def create_distilled_dataset(client: genai.Client, dataset: Dataset) -> Dataset:
    """Create a distilled dataset by querying the teacher for each question in the
    input dataset (using the `question` column)
    """
    # Store outputs from the teacher model (including reasoning and final answer)
    outputs: list[str] = []

    # TODO: Add your code here
    # ==== Start of your code ====
    with ThreadPoolExecutor() as executor:
        futures = [
            executor.submit(query_teacher, client, question, GEN_CONFIG)
            for question in dataset["question"]
        ]
        for future in tqdm.tqdm(as_completed(futures), total=len(futures)):
            outputs.append(future.result())
    # ==== End of your code ====

    completion = [[{"role": "assistant", "content": output}] for output in outputs]
    dataset = dataset.add_column("completion", completion)
    return dataset


if os.path.exists("distilled_gsm8k") and os.path.isdir("distilled_gsm8k"):
    print("Loading distilled dataset from disk...")
    distilled_dataset = DatasetDict.load_from_disk("distilled_gsm8k")
    train_distilled_dataset = distilled_dataset["train"]
    test_distilled_dataset = distilled_dataset["test"]
else:
    client = genai.Client()
    train_distilled_dataset = create_distilled_dataset(client, train_dataset)
    test_distilled_dataset = create_distilled_dataset(client, test_dataset)
    DatasetDict(
        {"train": train_distilled_dataset, "test": test_distilled_dataset}
    ).save_to_disk("distilled_gsm8k")


100%|██████████| 30/30 [01:04<00:00,  2.15s/it]


Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/30 [00:00<?, ? examples/s]

In [18]:
def filter_sample(sample: dict[str, Any]) -> bool:
    """Filter out samples based on the teacher's output quality (`completion` column),
    e.g., by checking the correctness of the final answer"""
    # TODO(Optional): Add your code here
    # ==== Start of your code ====
    return True
    # ==== End of your code ====


train_distilled_dataset = train_distilled_dataset.filter(filter_sample)

Filter:   0%|          | 0/300 [00:00<?, ? examples/s]

Evaluate the teacher model's performance

In [19]:
outputs = [
    completion[-1]["content"] for completion in test_distilled_dataset["completion"]
]
answers = [extract_final_answer(output) for output in outputs]
em = evaluate.load("exact_match")
em_score = em.compute(
    predictions=answers, references=test_distilled_dataset["ground_truth"]
)
print(f"Teacher's Exact Match: {em_score['exact_match']:.2f}")

Teacher's Exact Match: 0.33


## 2.2 SFT configurations

Next, let's set the configurations/hyperparameters for SFT.

**TODO:**
1. Modify the LoRA and SFT configurations so that the distilled model achieves the desired quality.

In [20]:
# LoRA config
# TODO: Modify these hyperparameters
# ==== Start of your code ====
lora_r = 32
lora_alpha = 32
lora_dropout = 0.1
target_modules = ["q_proj", "v_proj", "k_proj"]
# ==== End of your code ====

In [21]:
# SFT config
student_model_name = "qwen-distilled-lora-sft"
# TODO: Modify these hyperparameters
# ==== Start of your code ====
num_train_epochs = 1
max_steps = -1
per_device_train_batch_size = 8
# Hyperparameters
optim = "paged_adamw_8bit"
gradient_accumulation_steps = 1
learning_rate = 5e-4
weight_decay = 0.01
lr_scheduler_type = "cosine"
warmup_steps = 10
# Other training arguments
logging_steps = 5
eval_steps = logging_steps
save_steps = 0
eval_strategy = "steps"
# ==== End of your code ====

In [22]:
# Create PEFT and SFT configs
# NOTE: We do not recommend modifying this block.
peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)
sft_args = SFTConfig(
    num_train_epochs=num_train_epochs,
    max_steps=max_steps,
    per_device_train_batch_size=per_device_train_batch_size,
    # Training hyperparameters
    optim=optim,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    lr_scheduler_type=lr_scheduler_type,
    warmup_steps=warmup_steps,
    # Other training arguments
    logging_steps=logging_steps,
    eval_steps=eval_steps,
    save_steps=save_steps,
    eval_strategy=eval_strategy,
    # Fixed training arguments for efficient fine-tuning
    output_dir=student_model_name,
    gradient_checkpointing=True,
    completion_only_loss=True,
    bf16=torch_dtype == torch.bfloat16,
    group_by_length=True,
)
# bitsandbytes config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
)

## 2.3 Finetune the student model

Fine-tune the quantized student model on the distilled dataset.

In [23]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    dtype=torch_dtype,
    device_map="auto",
    use_cache=False,
)
model = prepare_model_for_kbit_training(model)
trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_distilled_dataset,
    eval_dataset=test_distilled_dataset,
    peft_config=peft_config,
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

In [24]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
5,1.151811,1.123703
10,1.050436,0.986095
15,0.955005,0.923602
20,1.116661,0.904878
25,0.970502,0.881200
30,0.840849,0.898421
35,0.949642,0.886396


TrainOutput(global_step=38, training_loss=1.0072020355023836, metrics={'train_runtime': 336.818, 'train_samples_per_second': 0.891, 'train_steps_per_second': 0.113, 'total_flos': 486411326410752.0, 'train_loss': 1.0072020355023836})

In [25]:
trainer.evaluate()

{'eval_loss': 0.8883534073829651,
 'eval_runtime': 11.5461,
 'eval_samples_per_second': 2.598,
 'eval_steps_per_second': 0.346}

In [26]:
trainer.model.save_pretrained(student_model_name)

In [27]:
# Free up GPU memory
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

See how the fine-tuned model performs.

In [28]:
model = AutoPeftModelForCausalLM.from_pretrained(
    student_model_name, quantization_config=bnb_config, device_map="auto"
)
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
sample = test_distilled_dataset[0]
prompt = sample["prompt"]
output = generator(prompt, generation_config=GEN_CONFIG)
del model
del generator
print(f"Prompt:\n{prompt}")
print("-" * 50)
print(f"Output:\n{output[0]['generated_text'][-1]['content']}")
print("-" * 50)
print(f"Expected:\n{sample['completion'][0]['content']}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Prompt:
[{'role': 'system', 'content': 'You are a helpful assistant for solving math word problems.'}, {'role': 'user', 'content': 'Answer the following question by explaining your reasoning very concisely and then end your answer with the final numerical answer prefixed by "#### ".\n\nDarrell and Allen\'s ages are in the ratio of 7:11. If their total age now is 162, calculate Allen\'s age 10 years from now.\n'}]
--------------------------------------------------
Output:
First find the total number of people: 3 + 4 = 7.
Then divide the total cost by the number of people to get the cost per person: $58 / 7 = $8.29.

#### 8.29
--------------------------------------------------
Expected:
Indras has 6 letters. Half of that is 3. Her sister has 4 more than 3, which is 7. Together, they have 6 + 7 = 13 letters.

#### 13


## 2.4 Evaluate the student model (20pt)

Below is the grading criteria:

- `Exact Match`
   - **20**: >= 0.30
   - **10**: >= 0.15 and < 0.30
   - **0**: < 0.15

In [29]:
model = AutoPeftModelForCausalLM.from_pretrained(
    student_model_name, quantization_config=bnb_config, device_map="auto"
)
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
outputs: list[str] = []
for i in tqdm.tqdm(range(0, len(test_distilled_dataset), GEN_BATCH_SIZE)):
    batch = test_distilled_dataset[i : i + GEN_BATCH_SIZE]
    prompts = batch["prompt"]
    output = generator(prompts, generation_config=GEN_CONFIG, batch_size=GEN_BATCH_SIZE)
    outputs.extend([o[0]["generated_text"][-1]["content"] for o in output])

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:55<00:00, 27.88s/it]


In [30]:
answers = [extract_final_answer(output) for output in outputs]
em = evaluate.load("exact_match")
em_score = em.compute(
    predictions=answers, references=test_distilled_dataset["ground_truth"]
)
print(f"Exact Match: {em_score['exact_match']:.2f}")

Exact Match: 0.13


In [31]:
# Free up memory
del generator
del model
gc.collect()
torch.cuda.empty_cache()

# 3. LLM Cascading

Lastly, we are going to implement an LLM cascade to reduce inference cost.

We will use the student model as the weaker model and the teacher model as the stronger model. For the routing signal, we will use the weaker model's **answer consistency**, defined as the ratio of the most consistent answer (by majority voting) among the multiple output samples. We refer to this ratio as the student's **confidence score**.

The LLM cascade should follow the steps below
1. Sample multiple outputs from the weaker model, given the input question.
2. Extract the final answers from the sampled outputs using the `extract_final_answer` function.
3. Identify the most consistent answer and calculate the confidence score.
4. Return the answer if the confidence score is higher than a pre-defined threshold; otherwise, forward the question to the stronger model.

In [32]:
from transformers import PreTrainedModel

## 3.1 Confidence scoring (20pt)

Next, we will implement the function for calculating the student's answer and confidence score.

**TODO:**
1. Implement the `student_answer_and_confidence` function to compute the student's answer and confidence score for the given question. See the definition of the confidence score above.

In [33]:
# TODO(Optional): Import libraries, define constants, and implement helper functions here
# ==== Start of your code ====
# Assuming 'generator' and 'extract_final_answer' are available in the scope
# The 'generator' pipeline is instantiated later in the notebook, so I'll create it locally for this function
# Or, rather, I should pass it as an argument or ensure it's globally accessible.
# Given the structure, it's likely intended to be used with the pipeline setup that wraps the model and tokenizer.
# I will modify the function signature to accept the pipeline directly, or create one inside if the model/tokenizer are passed.
# Looking at the original cell: 'generator = pipeline("text-generation", model=model, tokenizer=tokenizer)'
# It seems 'generator' is the best way to interact with the model within the pipeline context.
# For now, I will assume a global 'tokenizer' and a generator created from 'model' for this function to work.
# Alternatively, if `model` is passed, we can create a local generator.
# Let's use the provided `model` and `tokenizer` to create a local generator for sampling.
# ==== End of your code ====


def student_answer_and_confidence(
    model: PreTrainedModel,
    messages: list[dict[str, str]],
    generation_config: GenerationConfig,
    num_samples: int,
) -> tuple[str, float]:
    """Given a student model and input messages, return the student's answer and
    confidence score

    The confidence score is calculated by sampling multiple answers from the student
    model and calculating the vote share of the most consistent answer.

    Parameters
    ----------
    model: PreTrainedModel
        The student model to query
    messages: list[dict[str, str]]
        The messages to send to the student model
    generation_config: GenerationConfig
        The generation config to use when sampling answers from the student model
    num_samples: int
        The number of samples to draw from the student model

    Returns
    -------
    answer: str
        The student's answer to the question (the final answer extracted from the
        model's output)
    confidence: float
        The student's confidence
    """
    # TODO: Add your code here
    # ==== Start of your code ====
    local_gen_config = GenerationConfig(
        max_new_tokens=generation_config.max_new_tokens,
        do_sample=generation_config.do_sample,
        temperature=generation_config.temperature,
        top_p=generation_config.top_p,
        top_k=generation_config.top_k,
        num_return_sequences=num_samples,
        use_cache=generation_config.use_cache,
    )

    # Create a generator pipeline for the student model
    # Assuming tokenizer is globally available or can be passed
    local_generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

    outputs = local_generator(messages, generation_config=local_gen_config)

    extracted_answers = []
    for o in outputs:
        # Assuming the structure is similar to previous outputs, where the last item in 'generated_text' is the relevant content
        if isinstance(o['generated_text'], list) and o['generated_text'] and 'content' in o['generated_text'][-1]:
            extracted_answers.append(extract_final_answer(o['generated_text'][-1]['content']))
        elif isinstance(o['generated_text'], str):
             extracted_answers.append(extract_final_answer(o['generated_text']))
        else:
             # Handle cases where output structure might differ, e.g., if it's a direct string
            extracted_answers.append(extract_final_answer(str(o['generated_text'])))

    if not extracted_answers:
        return "", 0.0

    # Count occurrences of each answer
    from collections import Counter
    answer_counts = Counter(extracted_answers)

    # Find the most common answer and its count
    most_common_answer, count = answer_counts.most_common(1)[0]

    # Calculate confidence score
    confidence = count / num_samples

    # Free up local generator memory
    del local_generator
    gc.collect()
    torch.cuda.empty_cache()

    return most_common_answer, confidence
    # ==== End of your code ====


## 3.2 Cascade policy (30pt)

Now, we will implement the LLM cascade using the student and teacher models. We provide a basic implementation of a cascade below. Your role is to adjust the cascade's parameters to achieve the desired output quality and cost reduction.

**NOTE:**
- We do not call the teacher model directly in this part. Instead, we use the outputs of the teacher model previously collected in Section 2.1 and simulate the API cost.

**TODO:**
1. Tune the cascade's parameters below.

**Grading criteria:**
1. `Overall EM`
    - **15**: ≥ 0.6
    - **7.5**: ≥ 0.3 and < 0.6
    - **0**: < 0.3
2. `Cost savings`
    - **15**: ≥ 0.3
    - **7.5**: ≥ 0.15 and < 0.3
    - **0**: < 0.15

In [34]:
# TODO: Tune the following parameters
# ==== Start of your code ====
STUDENT_GEN_CONFIG = GenerationConfig(
    max_new_tokens=GEN_CONFIG.max_new_tokens,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    use_cache=True,
)
STUDENT_NUM_SAMPLES = 5
STUDENT_CONFIDENCE_THRESHOLD = 0.5
# ==== End of your code ====

In [35]:
STUDENT_COST_PER_SAMPLE = 1.0
TEACHER_COST_PER_CALL = 70.0


def llm_cascade(
    student_model: PreTrainedModel, sample: dict[str, Any]
) -> tuple[str, dict[str, Any]]:
    """Answer the question with an LLM cascade

    Returns
    -------
    answer: str
        The final answer to the question
    info: dict
        A dictionary containing additional information
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": user_prompt_template.format(question=sample["question"]),
        },
    ]
    student_answer, student_confidence = student_answer_and_confidence(
        student_model, messages, STUDENT_GEN_CONFIG, STUDENT_NUM_SAMPLES
    )
    info = {
        "student_num_samples": STUDENT_NUM_SAMPLES,
        "student_answer": student_answer,
        "student_confidence": student_confidence,
        "student_cost": STUDENT_COST_PER_SAMPLE * STUDENT_NUM_SAMPLES,
        "teacher_answer": None,
        "teacher_cost": 0,
    }
    if student_confidence >= STUDENT_CONFIDENCE_THRESHOLD:
        return student_answer, info
    teacher_output = sample["completion"][-1]["content"]  # Mock teacher model call
    teacher_answer = extract_final_answer(teacher_output)
    info["teacher_answer"] = teacher_answer
    info["teacher_cost"] = TEACHER_COST_PER_CALL
    return teacher_answer, info


def evaluate_llm_cascade(
    student_model: PreTrainedModel, dataset: Dataset
) -> dict[str, Any]:
    """Evaluate the LLM cascade on the given dataset

    Returns
    -------
    results: dict[str, Any]
         Evaluation results
    """
    predictions: list[str] = []
    references: list[str] = []
    infos: list[dict[str, Any]] = []
    for sample in tqdm.tqdm(dataset):
        reference = sample["ground_truth"]
        prediction, info = llm_cascade(student_model, sample)
        predictions.append(prediction)
        references.append(reference)
        infos.append(info)

    # Exact match scores
    em = evaluate.load("exact_match")
    overall_exact_match = em.compute(predictions=predictions, references=references)[
        "exact_match"
    ]
    student_exact_match = em.compute(
        predictions=[info["student_answer"] for info in infos], references=references
    )["exact_match"]

    # Inference costs
    total_student_cost = sum(info["student_cost"] for info in infos)
    total_teacher_cost = sum(info["teacher_cost"] for info in infos)
    total_cost = total_student_cost + total_teacher_cost
    teacher_only_cost = TEACHER_COST_PER_CALL * len(dataset)
    cost_savings = 1 - (total_cost / teacher_only_cost)

    # Other metrics
    avg_student_confidence = np.mean([info["student_confidence"] for info in infos])
    total_student_samples = sum(info["student_num_samples"] for info in infos)
    is_teacher_called = [
        1 if info["teacher_answer"] is not None else 0 for info in infos
    ]
    total_teacher_calls = sum(is_teacher_called)
    teacher_call_rate = np.mean(is_teacher_called)

    return {
        "overall_exact_match": float(overall_exact_match),
        "student_exact_match": float(student_exact_match),
        "total_student_cost": total_student_cost,
        "total_teacher_cost": total_teacher_cost,
        "total_cost": total_cost,
        "teacher_only_cost": teacher_only_cost,
        "cost_savings": cost_savings,
        "avg_student_confidence": float(avg_student_confidence),
        "total_student_samples": total_student_samples,
        "total_teacher_calls": total_teacher_calls,
        "teacher_call_rate": float(teacher_call_rate),
    }

In [36]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
)
student_model = AutoPeftModelForCausalLM.from_pretrained(
    student_model_name, quantization_config=bnb_config, device_map="auto"
).merge_and_unload()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [37]:
cascade_results = evaluate_llm_cascade(student_model, test_distilled_dataset)
cascade_results

  3%|▎         | 1/30 [01:13<35:40, 73.80s/it]


KeyboardInterrupt: 

In [ ]:
print("Summary of results:")
print(f"  Overall EM: {cascade_results['overall_exact_match']:.4f}")
print(f"  Student EM: {cascade_results['student_exact_match']:.4f}")
print(f"  Total student cost: {cascade_results['total_student_cost']:.4f}")
print(f"  Total teacher cost: {cascade_results['total_teacher_cost']:.4f}")
print(f"  Total inference cost: {cascade_results['total_cost']:.4f}")
print(f"  Teacher-only inference cost: {cascade_results['teacher_only_cost']:.4f}")
print(f"  Cost savings: {cascade_results['cost_savings']:.4f}")

In [ ]:
# Free up memory
del student_model
gc.collect()
torch.cuda.empty_cache()